# BDC 2026 - Cek Kemiripan Visual: Anomali Test vs Data Train Recyclable

**Tidak menambah gambar apapun** ke folder train -- notebook ini murni mencari tahu, di antara
gambar-gambar `0_Recyclable` yang SUDAH ADA, apakah ada yang secara visual mirip dengan 31 gambar
anomali test yang sudah kamu identifikasi manual (kreasi kertas/plastik/tekstil berbentuk/berwarna
organik).

**Caranya:** pakai model SigLIP2 yang sudah kamu latih sebagai *feature extractor* (bukan untuk
klasifikasi, tapi ambil representasi vektornya sebelum layer klasifikasi terakhir), lalu cari
tetangga terdekat (cosine similarity) antara tiap gambar anomali dan seluruh gambar `0_Recyclable`
di training set.

**Dua kemungkinan hasil, dan artinya:**
- **Ada tetangga yang mirip & berlabel benar (Recyclable)** -> polanya sebenarnya ADA di training
  set, tapi model gagal memanfaatkannya. Solusinya di ranah *training* (oversampling/reweighting
  gambar itu, bukan nambah data baru) -- lihat bagian solusi di akhir chat.
- **Tidak ada tetangga yang cukup mirip** (similarity jauh lebih rendah dari rata-rata) -> training
  set memang tidak punya contoh untuk pola ini, model wajar kesulitan generalisasi. Solusinya lebih ke
  arah *post-processing* / kalibrasi, bukan training.

In [1]:
import os

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

C:\Users\MyPC PRO\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

In [2]:
CONFIG = {
    "root": "BDC 2026",
    "model_name": "vit_base_patch16_siglip_224.v2_webli",
    "checkpoint": "vit_base_patch16_siglip_224.v2_webli_fold0.pth",  # ganti kalau nama file beda
    "img_size": 224,
    "norm": {"mean": (0.5, 0.5, 0.5), "std": (0.5, 0.5, 0.5)},
    "batch_size": 32,
    "num_workers": 0,
    "target_class_folder": "0_Recyclable",
    "top_k": 6,  # berapa tetangga terdekat yang ditampilkan per gambar anomali
    "output_dir": "similarity_check",
    # 31 id gambar test yang sudah kamu identifikasi manual sebagai kreasi/kerajinan
    "anomaly_test_ids": [
        27, 76, 116, 293, 312, 363, 372, 487, 499, 508, 637, 643, 661, 663, 728,
        761, 797, 843, 878, 899, 907, 973, 1033, 1035, 1079, 1092, 1144, 1168,
        1330, 1373, 1440,
    ],
}

## Dataset generik untuk feature extraction

In [3]:
class ImageListDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image = cv2.imread(path)
        if image is None:
            image = np.array(Image.open(path).convert("RGB"))
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform is not None:
            image = self.transform(image=image)["image"]
        return image, path


def get_transform(config):
    return A.Compose([
        A.Resize(config["img_size"], config["img_size"]),
        A.Normalize(mean=config["norm"]["mean"], std=config["norm"]["std"]),
        ToTensorV2(),
    ])

## Device & Model (dipakai sebagai feature extractor, BUKAN classifier)

`forward_features` + `forward_head(..., pre_logits=True)` mengambil representasi vektor gambar
SEBELUM masuk ke layer klasifikasi terakhir -- ini yang dipakai untuk cek kemiripan visual, bukan
prediksi kelasnya.

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

n_classes = 3
model = timm.create_model(CONFIG["model_name"], pretrained=False, num_classes=n_classes)
model.load_state_dict(torch.load(CONFIG["checkpoint"], map_location=device))
model.to(device)
model.eval()
print(f"Checkpoint dimuat: {CONFIG['checkpoint']}")


@torch.no_grad()
def extract_embeddings(image_paths, config, model, device, desc="Extracting"):
    transform = get_transform(config)
    dataset = ImageListDataset(image_paths, transform=transform)
    loader = DataLoader(dataset, batch_size=config["batch_size"], shuffle=False, num_workers=config["num_workers"])

    all_embeddings = []
    all_paths = []
    for images, paths in tqdm(loader, desc=desc):
        images = images.to(device)
        features = model.forward_features(images)
        pooled = model.forward_head(features, pre_logits=True)
        pooled = torch.nn.functional.normalize(pooled, dim=1)  # normalisasi -> cosine similarity = dot product
        all_embeddings.append(pooled.cpu().numpy())
        all_paths.extend(paths)

    return np.concatenate(all_embeddings, axis=0), all_paths

cuda
Checkpoint dimuat: vit_base_patch16_siglip_224.v2_webli_fold0.pth


## Ekstrak Embedding: Seluruh Gambar Train Recyclable

In [5]:
train_dir = os.path.join(CONFIG["root"], "train", CONFIG["target_class_folder"])
recyclable_paths = [os.path.join(train_dir, f) for f in os.listdir(train_dir)]
print(f"Total gambar {CONFIG['target_class_folder']}: {len(recyclable_paths)}")

recyclable_embeddings, recyclable_paths = extract_embeddings(
    recyclable_paths, CONFIG, model, device, desc="Recyclable train"
)
print(f"Selesai. Shape embedding: {recyclable_embeddings.shape}")

Total gambar 0_Recyclable: 9547


Recyclable train: 100%|██████████| 299/299 [01:12<00:00,  4.14it/s]

Selesai. Shape embedding: (9547, 768)


## Ekstrak Embedding: 31 Gambar Anomali Test

In [6]:
test_dir = os.path.join(CONFIG["root"], "test")
test_images = os.listdir(test_dir)
id_to_filename = {int("".join(filter(str.isdigit, f))): f for f in test_images}

anomaly_paths = []
missing_ids = []
for aid in CONFIG["anomaly_test_ids"]:
    fname = id_to_filename.get(aid)
    if fname is None:
        missing_ids.append(aid)
        continue
    anomaly_paths.append(os.path.join(test_dir, fname))

if missing_ids:
    print(f"PERINGATAN: id berikut tidak ditemukan di folder test: {missing_ids}")

anomaly_embeddings, anomaly_paths = extract_embeddings(
    anomaly_paths, CONFIG, model, device, desc="Anomali test"
)
print(f"Selesai. Shape embedding: {anomaly_embeddings.shape}")

Anomali test: 100%|██████████| 1/1 [00:00<00:00,  4.29it/s]

Selesai. Shape embedding: (31, 768)


## Cari Tetangga Terdekat (Cosine Similarity)

Karena embedding sudah dinormalisasi, cosine similarity = perkalian dot biasa (matrix multiply),
jadi ini murni operasi matriks, cepat walau datanya ribuan gambar.

In [7]:
# similarity_matrix[i, j] = cosine similarity antara anomali ke-i dan gambar recyclable train ke-j
similarity_matrix = anomaly_embeddings @ recyclable_embeddings.T

summary_rows = []
top_k = CONFIG["top_k"]

for i, apath in enumerate(anomaly_paths):
    sims = similarity_matrix[i]
    top_idx = np.argsort(sims)[::-1][:top_k]
    best_sim = sims[top_idx[0]]
    best_match = recyclable_paths[top_idx[0]]
    summary_rows.append({
        "anomaly_image": os.path.basename(apath),
        "best_match_similarity": best_sim,
        "best_match_file": os.path.basename(best_match),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("best_match_similarity", ascending=False)
print("=== Ringkasan: seberapa mirip tetangga TERDEKAT tiap anomali dengan data Recyclable train ===")
print(summary_df.to_string(index=False))

=== Ringkasan: seberapa mirip tetangga TERDEKAT tiap anomali dengan data Recyclable train ===
anomaly_image  best_match_similarity best_match_file
      312.jpg               0.881818      R_1560.jpg
      643.jpg               0.878948      R_1116.jpg
     1079.jpg               0.867311      R_1560.jpg
     1168.jpg               0.858662      R_1180.jpg
     1144.jpg               0.853315      R_1420.jpg
      363.jpg               0.852583       R_850.jpg
       76.jpg               0.850114      R_9992.jpg
     1035.jpg               0.847477      R_1180.jpg
      843.jpg               0.847400      R_1560.jpg
       27.jpg               0.844415       R_792.jpg
      907.jpg               0.844078      R_2652.jpg
      116.jpg               0.841305      R_4699.jpg
      973.jpg               0.835250      R_4709.jpg
      899.jpg               0.824914      R_2652.jpg
      499.jpg               0.823480      R_1420.jpg
     1330.jpg               0.823379      R_7659.jpg
     

## Baseline Pembanding: Similarity Internal Data Recyclable Sendiri

Supaya angka similarity di atas ada pembandingnya -- ambil sampel acak dari data Recyclable train,
dan hitung similarity tetangga terdekatnya SESAMA data Recyclable train (bukan ke anomali). Kalau
similarity tetangga-terdekat anomali JAUH LEBIH RENDAH dari baseline ini, itu bukti kuat bahwa
training set memang tidak punya pola serupa untuk gambar-gambar anomali tersebut.

In [8]:
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(recyclable_embeddings), size=min(200, len(recyclable_embeddings)), replace=False)

baseline_sims = []
for idx in sample_idx:
    sims = recyclable_embeddings @ recyclable_embeddings[idx]
    sims[idx] = -1  # exclude diri sendiri
    baseline_sims.append(sims.max())

baseline_sims = np.array(baseline_sims)

print("=== Perbandingan ===")
print(f"Similarity tetangga-terdekat ANOMALI (rata-rata)         : {summary_df['best_match_similarity'].mean():.4f}")
print(f"Similarity tetangga-terdekat data Recyclable NORMAL (baseline, rata-rata): {baseline_sims.mean():.4f}")
print(f"Selisih                                                   : {baseline_sims.mean() - summary_df['best_match_similarity'].mean():.4f}")

=== Perbandingan ===
Similarity tetangga-terdekat ANOMALI (rata-rata)         : 0.8192
Similarity tetangga-terdekat data Recyclable NORMAL (baseline, rata-rata): 0.9660
Selisih                                                   : 0.1467


## Visualisasi: Gambar Anomali + Tetangga Terdekatnya

Satu figure per gambar anomali, disimpan ke folder `similarity_check/`.

In [9]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)

for i, apath in enumerate(anomaly_paths):
    sims = similarity_matrix[i]
    top_idx = np.argsort(sims)[::-1][:top_k]

    fig, axes = plt.subplots(1, top_k + 1, figsize=((top_k + 1) * 2.3, 2.8))

    query_img = Image.open(apath)
    axes[0].imshow(query_img)
    axes[0].set_title(f"ANOMALI\n{os.path.basename(apath)}", fontsize=9, color="red")
    axes[0].axis("off")

    for j, idx in enumerate(top_idx):
        neighbor_img = Image.open(recyclable_paths[idx])
        axes[j + 1].imshow(neighbor_img)
        axes[j + 1].set_title(f"sim={sims[idx]:.3f}", fontsize=9)
        axes[j + 1].axis("off")

    plt.tight_layout()
    out_name = f"{CONFIG['output_dir']}/anomaly_{os.path.basename(apath).split('.')[0]}.png"
    plt.savefig(out_name, dpi=110)
    plt.close(fig)

print(f"Selesai. {len(anomaly_paths)} figure disimpan ke folder '{CONFIG['output_dir']}/'")
print("Buka folder itu untuk lihat satu per satu -- tiap file menampilkan 1 gambar anomali + tetangga terdekatnya di training set.")

Selesai. 31 figure disimpan ke folder 'similarity_check/'
Buka folder itu untuk lihat satu per satu -- tiap file menampilkan 1 gambar anomali + tetangga terdekatnya di training set.
